In [ ]:
from openai import OpenAI

#openai_client = OpenAI()
openai_client = OpenAI(
    api_key="",
    base_url="https://openrouter.ai/api/v1"
)

In [2]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response

Once upon a time in the whimsical land of Mistwood, where the trees whispered secrets and flowers danced in the wind, there lived a unicorn named Glitterhorn. With a coat that shimmered like the night sky and a horn that sparkled with every color of the rainbow, Glitterhorn was beloved by all.

But Glitterhorn had a peculiar quirk. Unlike the other unicorns who played in meadows or galloped through glittering streams, Glitterhorn had an insatiable curiosity about the humans who lived in the nearby village. He often snuck out at night to watch them from the shadows, hoping to understand their world.

One fateful night, Glitterhorn decided to venture closer than ever before. He arrived at a grand festival in the village, where everyone was celebrating with music, dancing, and laughter. Glitterhorn was captivated by the flickering lights and joyous sounds. As he stepped out from the trees, his shimmering coat caught the eyes of the villagers.

Gasps of wonder filled the air, and Glitterho

In [3]:
from pydantic import BaseModel, Field

In [4]:
class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [5]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

with openai_client.responses.stream(
    model="gpt-4o-mini",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response

{"title":"The Misadventures of a Unicorn Named Sparkle","subtitle":"A Cautionary Tale of Ignorance and Consequences","sections":[{"header":"Once Upon a Time in a Magical Forest","text":"In a vibrant forest filled with dazzling flowers and twinkling streams lived a young unicorn named Sparkle. Unlike the other unicorns, Sparkle had a wild streak. She loved to chase butterflies, play pranks on the forest animals, and boast about her speed and beauty."},{"header":"The Fateful Day","text":"One sunny morning, Sparkle overheard the wise old owl warning the other creatures of the forest about the Dark Woods—a mysterious area where strange things happened and no creature returned unchanged. Ignoring the warning, Sparkle, full of bravado, decided to venture into the Dark Woods to prove her bravery."},{"header":"Curiosity Turns Dangerous","text":"As Sparkle trotted deeper into the Dark Woods, she marveled at the eerie beauty of the gnarled trees and glowing mushrooms. Suddenly, she spotted a gli

In [6]:
story = response.output_parsed

In [7]:
print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

# The Misadventures of a Unicorn Named Sparkle
A Cautionary Tale of Ignorance and Consequences

## Once Upon a Time in a Magical Forest

In a vibrant forest filled with dazzling flowers and twinkling streams lived a young unicorn named Sparkle. Unlike the other unicorns, Sparkle had a wild streak. She loved to chase butterflies, play pranks on the forest animals, and boast about her speed and beauty.


## The Fateful Day

One sunny morning, Sparkle overheard the wise old owl warning the other creatures of the forest about the Dark Woods—a mysterious area where strange things happened and no creature returned unchanged. Ignoring the warning, Sparkle, full of bravado, decided to venture into the Dark Woods to prove her bravery.


## Curiosity Turns Dangerous

As Sparkle trotted deeper into the Dark Woods, she marveled at the eerie beauty of the gnarled trees and glowing mushrooms. Suddenly, she spotted a glimmering crystal deep in the underbrush. Ignoring everything she'd been told, she 

In [8]:
from jaxn import JSONParserHandler, StreamingJSONParser

In [9]:
raw_json = """
{"title":"The Misadventures of Sparkle, the Unicorn","subtitle":"A Tale of Trouble and Lessons","sections":[{"header":"Once Upon a Time","text":"In a magical land far, far away, there lived a young unicorn named Sparkle. She was known for her shimmering coat and dazzling horn, but she had one major flaw: a mischievous streak that often led to chaos."},{"header":"The Forbidden Forest","text":"One sunny morning, Sparkle overheard her friends talking about the Forbidden Forest, a place where no unicorn dared to go. They said it was filled with danger and dark magic. But Sparkle, intrigued by the thrill, decided to venture into the forest alone."},{"header":"A Wretched Encounter","text":"As she galloped deeper into the forest, Sparkle found herself surrounded by eerie shadows and ominous sounds. Suddenly, she stumbled upon a group of grumpy trolls who were not too happy to see an intruder. Instead of being scared, Sparkle decided to play a prank on them."},{"header":"The Troll Trap","text":"Using her magic, she conjured up illusions of beautiful flowers and sparkling jewels. The trolls, greedy and foolish, fell for her trick and rushed toward the illusions. However, in doing so, they knocked over a giant boulder, blocking Sparkle's way back."},{"header":"The Consequences","text":"Trapped and surrounded by trolls who were now angrier than ever, Sparkle realized her mischief had backfired. Her only hope was to escape the trolls and unblock the path, but her magic was drained from the prank."},{"header":"A Lesson Learned","text":"Just when things seemed bleak, Sparkle remembered her true friends who would come looking for her. She calmed herself, using her wit instead of magic. She negotiated with the trolls to let her go, promising never to trespass again."},{"header":"Home Safe, but Changed","text":"With the trolls distracted by a fine tale of treasures far beyond the forest, Sparkle dashed away and eventually found her way home. Though she escaped, she learned that her mischievous nature could lead her into great trouble. From that day on, Sparkle decided to be a better friend and use her magic wisely."},{"header":"The End","text":"And so, the tale of Sparkle the unicorn spread throughout the land, reminding everyone that sometimes, the thrill of adventure can come with consequences. And being true to oneself is the most magical lesson of all."}]}
""".strip()

In [10]:
from typing import Any, Dict

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)
    

In [11]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

In [12]:
for c in raw_json:
    parser.parse_incremental(c)

# The Misadventures of Sparkle, the Unicorn
A Tale of Trouble and Lessons


## Once Upon a Time

In a magical land far, far away, there lived a young unicorn named Sparkle. She was known for her shimmering coat and dazzling horn, but she had one major flaw: a mischievous streak that often led to chaos.

## The Forbidden Forest

One sunny morning, Sparkle overheard her friends talking about the Forbidden Forest, a place where no unicorn dared to go. They said it was filled with danger and dark magic. But Sparkle, intrigued by the thrill, decided to venture into the forest alone.

## A Wretched Encounter

As she galloped deeper into the forest, Sparkle found herself surrounded by eerie shadows and ominous sounds. Suddenly, she stumbled upon a group of grumpy trolls who were not too happy to see an intruder. Instead of being scared, Sparkle decided to play a prank on them.

## The Troll Trap

Using her magic, she conjured up illusions of beautiful flowers and sparkling jewels. The trolls,

In [13]:
def llm_structured_stream(
        user_prompt,
        output_type,
        parser_handler=JSONParserHandler(),
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response

In [14]:
instructions = "your task is to tell the user bad time stories"
user_prompt =  "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Dark Side of Unicorns
Not all that glitters is gold...


## The Last Unicorn of the Dark Forest

In a land not so far away, there was a dark forest where only the bravest adventurers dared to tread. Within this forest lived a unicorn, once radiant and filled with light, now twisted and corrupted by the darkness that surrounded it. This unicorn, known as the Shadowhorn, had a mane that shimmered with black glitter, and its horn, once pure, was now a vessel of nightmares.

Legend spoke of how the Shadowhorn was once a guardian of harmony and light, but when the darkness seeped into the forest, it consumed the unicorn's heart. It now roamed the forest, luring travelers with its beauty but leading them to eternal despair instead. Many who sought to capture this 'beautiful' creature fell into traps of thorns and shadows, never to return.

## The Curse of the Glittering Hooves

Another tale speaks of a young girl named Clara who wished upon a star to meet a unicorn. Her wish was grante

In [15]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results



instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""


prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Indexed 382 chunks from 95 documents


In [16]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [17]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)


In [18]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

In [19]:
response = rag('llm as a judge')

Using an LLM as a judge involves a structured process where an LLM (Large Language Model) is utilized to evaluate text responses based on specific criteria. This approach can be implemented in various ways:

1. **Reference-based Evaluation**: This method compares new responses against a set of approved, accurate responses. It is particularly useful for regression testing or situations where a "ground truth" exists.

2. **Open-ended Evaluation**: This method allows for the assessment of responses based on custom criteria, which is advantageous when no reference answers are available.

In this tutorial example, you create an evaluation dataset, design an LLM evaluator prompt, and compare the evaluations from the LLM judge against manual labels to assess quality. You will also set up a prompt that evaluates correctness and verbosity, helping to classify whether responses are accurate or succinct enough.

To implement this, you will need basic Python knowledge and an OpenAI API key. The tu